Build Multimodal RAG Embedding modal

STEP01: Import necessary packages


In [3]:
import fitz  # PyMuPDF
from langchain_core.documents import Document
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from sklearn.metrics.pairwise import cosine_similarity
import os
import base64
import io
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

/Users/alisultanomariv/Documents/Multimodal RAG Embeddings/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


STEP02: Initialize CLIP 

In [5]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

### initialize CLIP model and processor
clip_model=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()




Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

STEP03: Initialize Embeddings

In [12]:
# Image Embedding Function
def embed_image(image_data):
    """Generate image embeddings using CLIP model."""
    if isinstance(image_data, str):
        image = Image.open(image_data).convert("RGB")
    else: 
        image = image_data

    inputs=clip_processor(image=image, return_tensors="pt")
    with torch.no_grad():
        features = clip_model.get_image_features(**inputs
                                                 )
        # Normalize embeddings to unit vector
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().numpy()
# Text Embedding Function
def embed_text(text):
    """Generate text embeddings using CLIP model."""
    inputs = clip_processor(text=text, return_tensors="pt", padding=True, truncation=True, max_length=77)
    with torch.no_grad():
        features = clip_model.get_text_features(**inputs)
        # Normalize embeddings to unit vector
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().numpy()

STEP04: Add PDF processing

In [9]:
# Load PDF Document
pdf_path = "multimodal_sample.pdf"
docs = fitz.open(pdf_path)
#Storade for all documents and embeddings
all_docs = []
all_embeddings = []
image_data_store = {} 

# Text splitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)



In [10]:
docs

Document('multimodal_sample.pdf')

STEP05: Extract text and image from pdf and convert them into embeddings

In [13]:
for i, page in enumerate(docs):
    ##process text
    text=page.get_text()
    if text.strip():
        # Split text into chunks
        temp_docs = Document(page_content=text, metadata={"page":i, "type":"text"})
        text_chunks = text_splitter.split_documents([temp_docs])
        #Embed each chunk using CLIP
        for chunk in text_chunks:
            embedding = embed_text(chunk.page_content)
            all_embeddings.append(embedding)
            all_docs.append(chunk)


    ##process images
    for img_index, img in enumerate(page.get_images(full=True)):
        try:
            xref = img[0]
            base_image = docs.extract_image[xref]
            image_bytes = base_image["image"]
            ##Convert PDF image to PIL format
            pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
            ##Generate unique identifier
            image_id = f"page_{i}_img_{img_index}"


    

            ##Store as base64 for GPT-4V 
            buffered = io.BytesIO()
            pil_image.save(buffered, format="PNG")
            img_base64 = base64.b64tncode(buffered.getcalue()).decode()
            image_data_store[image_id] = img_base64

            ##Create CLIP embedding for retrieval
            embedding = embed_image(pil_image)
            all_embeddings.append(embedding)

            # Create Document for image
            image_doc = Document(page_content=f"[Image: {image_id}]", metadata={"page":1, "type":"image", "image_id": image_id})
            all_docs.append(image_doc)
        except Exception as e:
            print(f"Error processing image {img_index} on page {i}: {e}")
            continue
docs.close()




Error processing image 0 on page 0: 'method' object is not subscriptable


In [14]:
all_embeddings

[array([-2.67243385e-03,  1.28299333e-02, -5.18314540e-02,  4.14879769e-02,
        -2.33941991e-02, -7.55868386e-03, -3.67659070e-02,  1.19710773e-01,
         8.52081701e-02,  2.05413275e-03, -1.11533860e-02, -1.29592652e-02,
         5.25014475e-02, -3.65390303e-03,  4.76078242e-02,  1.58372745e-02,
         2.03387495e-02,  4.35361564e-02, -3.29174055e-03,  2.03181189e-02,
         1.88018696e-03, -4.23493795e-02,  5.44102304e-03,  3.70934717e-02,
        -1.65622793e-02,  6.48647314e-03, -4.78011817e-02,  8.67477432e-03,
         5.88859878e-02, -3.21394056e-02,  4.32439931e-02,  9.65298433e-03,
        -4.47922247e-03, -1.94857009e-02, -3.63503136e-02, -1.23472475e-02,
        -2.17928439e-02, -1.99016873e-02,  8.09620023e-02, -3.32987048e-02,
        -2.38900799e-02, -3.96138355e-02, -1.27279675e-02,  3.50380912e-02,
        -2.52217390e-02,  2.00031861e-03,  1.49660530e-02, -2.31977496e-02,
        -6.86791688e-02, -5.25775307e-04, -2.22545397e-02, -1.04103955e-02,
        -1.9

In [15]:
all_docs

[Document(metadata={'page': 0, 'type': 'text'}, page_content='Annual Revenue Overview\nThis document summarizes the revenue trends across Q1, Q2, and Q3. As illustrated in the chart\nbelow, revenue grew steadily with the highest growth recorded in Q3.\nQ1 showed a moderate increase in revenue as new product lines were introduced. Q2 outperformed\nQ1 due to marketing campaigns. Q3 had exponential growth due to global expansion.')]

STEP06: Store embeddings in vector database

In [17]:
# Initialize unified FAISS vector store with CLIP embeddings
embeddings_array = np.array(all_embeddings)
# Initialize FAISS index
vector_store=FAISS.from_embeddings(
    text_embeddings=[(docs.page_content, embedding) for docs, embedding in zip(all_docs, embeddings_array)],
    embedding=None,
    metadatas=[docs.metadata for docs in all_docs]
)

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


In [18]:
embeddings_array

array([[-2.67243385e-03,  1.28299333e-02, -5.18314540e-02,
         4.14879769e-02, -2.33941991e-02, -7.55868386e-03,
        -3.67659070e-02,  1.19710773e-01,  8.52081701e-02,
         2.05413275e-03, -1.11533860e-02, -1.29592652e-02,
         5.25014475e-02, -3.65390303e-03,  4.76078242e-02,
         1.58372745e-02,  2.03387495e-02,  4.35361564e-02,
        -3.29174055e-03,  2.03181189e-02,  1.88018696e-03,
        -4.23493795e-02,  5.44102304e-03,  3.70934717e-02,
        -1.65622793e-02,  6.48647314e-03, -4.78011817e-02,
         8.67477432e-03,  5.88859878e-02, -3.21394056e-02,
         4.32439931e-02,  9.65298433e-03, -4.47922247e-03,
        -1.94857009e-02, -3.63503136e-02, -1.23472475e-02,
        -2.17928439e-02, -1.99016873e-02,  8.09620023e-02,
        -3.32987048e-02, -2.38900799e-02, -3.96138355e-02,
        -1.27279675e-02,  3.50380912e-02, -2.52217390e-02,
         2.00031861e-03,  1.49660530e-02, -2.31977496e-02,
        -6.86791688e-02, -5.25775307e-04, -2.22545397e-0

In [19]:
vector_store

STEP07: Initialize GPT-4 Vision model

In [23]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("openai:gpt-4.1")

llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x17e9b4a50>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x17e9b4e10>, root_client=<openai.OpenAI object at 0x17e9b47d0>, root_async_client=<openai.AsyncOpenAI object at 0x17e9b4b90>, model_name='gpt-4.1', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

STEP08: Initialize query to embedding and comparison

In [24]:
def retireve_multimodal(query, k=5):
    """Unified retrieval for CLIP embeddings for both text and images."""
    # Embed query using CLIP
    query_embedding = embed_text(query)

    # Retrieve top-k similar embeddings from vector store based on cosine similarity with query embedding
    results = vector_store.similarity_search_by_vector(
        embedding=query_embedding,
        k=k

    
    )
    return results

STEP09: Initialize GPT-4.1 response

In [ ]:
def create_multimpdal_message(query, retrieved_docs):
    """Create a message for GPT-4 with text and image data."""
    content=[]

    # Add the query
    content.append({
        "type": "text",
        "text": f"Question: {query}\n\nContext:\n"
    })

    # Separate text and image contents
    text_docs = [docs for docs in retrieved_docs if dpcs.metadata.get("type") == "text"]
    image_docs = [docs for docs in retrieved_docs if docs.metadata.get("type") == "image"]


    # Add text context
    if text_docs:
        text_context = "\n\n".join([
            f"[Page {docs.metadata['page']}]: {docs.page_content}"
            for docs in text_docs
        ])
        content.append({
            "type": "text",
            "text": text_context
        })

    # Add image context
    for docs in image_docs:
        image_id = docs.metadata("image_id")
        if image_id and image_id in image_data_store:
            content.append({
                "type": "text",
                "text": f"\n[Image from page {docs.metadata['page']}]:\n"
            })
            content.append({
                "type": "image_url",
                "image_url": {
                "url": f"data:image/png;base64,{image_data_store[image_id]}"
                }
            })
    
    # Add instructions
    content.append({
        "type": "text",
        "text": "\n\nAnswer the question based on the provided text and images."


    })

    return HumanMessage(content=content)
